# Create Chan Zuckerberg Initiative (CZI) Grants (GRANT PATTERN, method-5 static HTML)

CZI grants from the foundation's WordPress at chanzuckerberg.com. Major US funder of biomedicine, education, and open science, including Human Cell Atlas, EOSS (open-source software), Imaging, Neurodegenerative Disease Networks, Pediatric Disease, and others.

**Prerequisites:**
- Run `scripts/local/czi_to_s3.py` first.

**Data source:** 16 WordPress YOAST sub-sitemaps (`/czi_eoss-sitemap.xml`, `/czi_hca-sitemap.xml`, etc.) — total **~870 grant URLs** across 16 distinct funding programs:

| program key | label | URLs |
|-------------|-------|------|
| czi_eoss | Essential Open Source Software for Science | 195 |
| czi_ncn | Neurodegenerative Disease Networks | 133 |
| czi_hca | Human Cell Atlas | 123 |
| czi_frontiers | Frontiers of Imaging | 76 |
| czi_imaging | Imaging | 74 |
| czi_los | Listening to Our Stakeholders | 57 |
| czi_data_insights | Data Insights | 55 |
| czi_seed_networks | Seed Networks | 38 |
| czi_rfa | RFA | 32 |
| czi_single_cell | Single Cell Biology | 29 |
| czi_pediatric | Pediatric Disease | 18 |
| czi_rao | Rare As One | 17 |
| czi_ecb | Education Capacity Building | 6 |
| czi_sharing | Open Science Sharing | 6 |
| czi_cop | Communities of Practice | 6 |
| czi_other_grants | Other Grants | 5 |

Each grant page exposes:
- `<h1>` project title
- "Key Personnel | PI Name | Affiliation | Institution" prose
- "Funding Cycle | N", "Proposal Summary | ..."
- JSON-LD WebPage with `datePublished`

**S3 location:** `s3a://openalex-ingest/awards/czi/czi_grants.parquet`

**Awarding body:**
- funder_id: 4320315474
- display_name: "Chan Zuckerberg Initiative"
- country: US
- ROR: none
- DOI: 10.13039/100014989

**Coverage (smoke test 15 grants, 2026-05-23):**
- 100% title / slug / program_label / pi_raw / pi_affiliation / start_year / description

**Amount/currency:** NULL with §6.7 waiver. CZI publishes program-level budget envelopes (e.g., EOSS cycles allocate ~$1M total split across proposals) but does NOT publish per-proposal amount on individual grant pages. Grant/research-funder precedent: HHMI #44, CIFAR #79, Damon Runyon #73, Packard #95, Rita Allen #107, Schmidt Sciences #108, NOMIS #109, Wenner-Gren #110, Mercator #116, Fritz Thyssen #117.

**Provenance:** `czi_grants`.

**Priority:** 120.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.czi_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/czi/czi_grants.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.czi_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.czi_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'czi_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'czi_raw'
  AND LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT title, program_label, pi_raw, pi_affiliation, start_year,
       SUBSTR(description, 1, 150) AS desc_preview
FROM openalex.awards.czi_raw LIMIT 5;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.czi_raw;


In [ ]:
%sql
SELECT start_year, COUNT(*) as grants
FROM openalex.awards.czi_raw
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 25;


In [ ]:
%sql
SELECT program_label, COUNT(*) as grants
FROM openalex.awards.czi_raw
GROUP BY program_label ORDER BY grants DESC;


In [ ]:
%sql
-- Top PI affiliations
SELECT pi_affiliation, COUNT(*) as cnt
FROM openalex.awards.czi_raw
GROUP BY pi_affiliation ORDER BY cnt DESC LIMIT 15;


## Step 1.6: Fail-fast — Verify the CZI Funder Row Exists

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320315474;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.czi_awards
USING delta
AS
WITH
czi_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320315474
),
raw_prepared AS (
    SELECT
        *,
        TRY_CAST(start_year AS INT) AS parsed_year,
        TRY_TO_DATE(CONCAT(start_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date
    FROM openalex.awards.czi_raw
    WHERE title IS NOT NULL
      AND TRIM(title) != ''
      AND funder_award_id IS NOT NULL
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.title as display_name,
        g.description as description,

        f.funder_id,
        g.funder_award_id,

        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'research' as funding_type,
        COALESCE(NULLIF(TRIM(g.program_label), ''), 'CZI Grant') as funder_scheme,

        'czi_grants' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        struct(
            NULLIF(TRIM(g.pi_given_name), '') as given_name,
            NULLIF(TRIM(g.pi_family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.pi_affiliation), '') as name,
                'US' as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN czi_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 120

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'czi_grants' AND priority = 120;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    120 as priority
FROM openalex.awards.czi_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.czi_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.czi_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_start_date,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) as pct_pi
FROM openalex.awards.czi_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, funder_scheme,
       lead_investigator.given_name AS pi_given,
       lead_investigator.family_name AS pi_family,
       lead_investigator.affiliation.name AS pi_affiliation,
       landing_page_url
FROM openalex.awards.czi_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.czi_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.czi_awards
GROUP BY funder_scheme ORDER BY cnt DESC;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.czi_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 20;


In [ ]:
%sql
-- §6.7 amount/currency check (waivered)
SELECT COUNT(*) AS total, COUNT(amount) AS has_amount,
       ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
       COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.czi_awards;


In [ ]:
%sql
SELECT COUNT(*) AS total, COUNT(DISTINCT funder_award_id) AS distinct_ids
FROM openalex.awards.czi_awards;
